# Week 7: Evaluation Framework

Phase 1 마무리 - Pre-Freeze 평가

## 목표
1. **RAGAS 4지표 측정**: Faithfulness, Answer Relevancy, Context Precision, Context Recall
2. **도메인 특화 메트릭**: Refusal Accuracy, Citation Accuracy
3. **RAGAS 한계 사례 분석**: 점수와 실제 품질의 괴리 발굴

## 데이터셋
- Golden Set v1: `data/eval/golden_set_v1.csv` (35 questions, 5 q_types)

In [ ]:
import sys
sys.path.insert(0, '..')

import csv
import json
import re
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict

import pandas as pd
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.documents import Document

## 1. Load Golden Set v1

In [ ]:
@dataclass
class GoldenQuestion:
    """Golden Set 질문."""
    question: str
    ground_truth: str
    reference_context: str
    q_type: str
    modality_label: str
    notes: str

def load_golden_set(path: Path) -> list[GoldenQuestion]:
    """Load Golden Set v1 from CSV."""
    questions = []
    with open(path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            questions.append(GoldenQuestion(
                question=row['question'],
                ground_truth=row['ground_truth'],
                reference_context=row['reference_context'],
                q_type=row['q_type'],
                modality_label=row['modality_label'],
                notes=row['notes'],
            ))
    return questions

GOLDEN_SET_PATH = Path('../data/eval/golden_set_v1.csv')
golden_questions = load_golden_set(GOLDEN_SET_PATH)
print(f"Loaded {len(golden_questions)} questions")

# Distribution by q_type
q_type_counts = defaultdict(int)
for q in golden_questions:
    q_type_counts[q.q_type] += 1
print(f"\nBy q_type: {dict(q_type_counts)}")

## 2. Setup Baseline Retriever (Hybrid + Rerank)

In [ ]:
from src.vectorstore import load_vectorstore
from src.retrieval import HybridRerankerRetriever, HybridRetrieverConfig, RerankConfig

CHROMA_DIR = Path('../data/chroma_db_c3')

print("Loading vectorstore...")
vs = load_vectorstore(CHROMA_DIR, collection_name="lg_manuals_c3")
print(f"Loaded {vs._collection.count()} documents")

print("\nCreating Hybrid+Rerank retriever...")
retriever = HybridRerankerRetriever(
    vs,
    hybrid_config=HybridRetrieverConfig(bm25_weight=0.5, dense_weight=0.5),
    rerank_config=RerankConfig(first_stage_k=20, final_k=5),
)
print("Retriever ready.")

## 3. RAG Pipeline with Citation

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

RAG_PROMPT = """다음 컨텍스트를 바탕으로 질문에 답하세요.
반드시 출처(문서명, 페이지)를 명시하세요.
컨텍스트에서 답을 찾을 수 없으면 "제공된 문서에서 확인할 수 없습니다."라고 답하세요.

컨텍스트:
{context}

질문: {question}

답변:"""

def run_rag(question: str, retriever, llm) -> tuple[str, list[Document]]:
    """Run RAG pipeline with citation tracking."""
    docs = retriever.invoke(question)
    
    # Build context with source info
    context_parts = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'unknown')
        page = doc.metadata.get('page', '?')
        context_parts.append(f"[출처: {source} p.{page}]\n{doc.page_content}")
    
    context = "\n\n".join(context_parts)
    prompt = RAG_PROMPT.format(context=context, question=question)
    
    response = llm.invoke(prompt)
    return response.content, docs

# Test
test_q = golden_questions[0]
response, docs = run_rag(test_q.question, retriever, llm)
print(f"Q: {test_q.question}")
print(f"A: {response[:200]}...")

## 4. RAGAS 4지표 측정

In [ ]:
# RAGAS imports with monkey patch for version compatibility
from unittest.mock import MagicMock
_fake_vertexai = MagicMock()
_fake_vertexai.ChatVertexAI = MagicMock()
sys.modules["langchain_community.chat_models.vertexai"] = _fake_vertexai

from datasets import Dataset
from langchain_openai import OpenAIEmbeddings
from ragas import evaluate
from ragas.metrics import Faithfulness, ResponseRelevancy, ContextPrecision, ContextRecall

from src.ragas_helpers import merge_ragas_scores

# Explicit embedding/LLM wrappers: ragas 0.4.3's auto-instantiated
# ragas.embeddings.openai_provider.OpenAIEmbeddings lacks embed_query,
# which the new ResponseRelevancy metric calls. Passing langchain objects
# routes ragas through LangchainEmbeddingsWrapper, which exposes embed_query.
ragas_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


In [ ]:
def evaluate_with_ragas(
    questions: list[GoldenQuestion],
    retriever,
    llm,
    sample_size: int | None = None,
) -> pd.DataFrame:
    """Run RAGAS evaluation on golden set."""
    # Filter: exclude out_of_scope (no context needed)
    eval_questions = [q for q in questions if q.q_type != 'out_of_scope']

    if sample_size:
        eval_questions = eval_questions[:sample_size]

    print(f"Evaluating {len(eval_questions)} questions...")

    results = []
    for i, q in enumerate(eval_questions):
        print(f"  [{i+1}/{len(eval_questions)}] {q.question[:30]}...")

        response, docs = run_rag(q.question, retriever, llm)
        contexts = [doc.page_content for doc in docs]

        results.append({
            'question': q.question,
            'q_type': q.q_type,
            'ground_truth': q.ground_truth,
            'reference_context': q.reference_context,
            'response': response,
            'contexts': contexts,
            'docs': docs,  # Keep for citation accuracy
        })

    # Build RAGAS dataset
    ragas_data = {
        'user_input': [r['question'] for r in results],
        'response': [r['response'] for r in results],
        'retrieved_contexts': [r['contexts'] for r in results],
        'reference': [r['ground_truth'] for r in results],
    }
    dataset = Dataset.from_dict(ragas_data)

    # Evaluate
    print("\nRunning RAGAS evaluation...")
    metrics = [Faithfulness(), ResponseRelevancy(), ContextPrecision(), ContextRecall()]
    ragas_result = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=llm,
        embeddings=ragas_embeddings,
    )

    # Merge results (handles ragas 0.4.3 `answer_relevancy` → `response_relevancy`)
    ragas_df = ragas_result.to_pandas()
    results = merge_ragas_scores(results, ragas_df)

    return pd.DataFrame(results)


In [ ]:
# Run RAGAS evaluation (sample for time constraint)
# Set sample_size=None for full evaluation
ragas_results = evaluate_with_ragas(golden_questions, retriever, llm, sample_size=10)

# Summary statistics
print("\n=== RAGAS 4지표 Summary ===")
print(f"Faithfulness:       {ragas_results['faithfulness'].mean():.3f}")
print(f"Response Relevancy: {ragas_results['response_relevancy'].mean():.3f}")
print(f"Context Precision:  {ragas_results['context_precision'].mean():.3f}")
print(f"Context Recall:     {ragas_results['context_recall'].mean():.3f}")

In [ ]:
# By q_type breakdown
print("\n=== By q_type ===")
for q_type in ragas_results['q_type'].unique():
    subset = ragas_results[ragas_results['q_type'] == q_type]
    print(f"\n{q_type} (n={len(subset)}):")
    print(f"  Faithfulness:       {subset['faithfulness'].mean():.3f}")
    print(f"  Response Relevancy: {subset['response_relevancy'].mean():.3f}")
    print(f"  Context Precision:  {subset['context_precision'].mean():.3f}")
    print(f"  Context Recall:     {subset['context_recall'].mean():.3f}")

## 5. Refusal Accuracy

In [ ]:
REFUSAL_PATTERNS = [
    r"제공된 문서에서 확인할 수 없습니다",
    r"문서에서 확인할 수 없",
    r"정보를 찾을 수 없",
    r"해당 정보가 없",
    r"포함되어 있지 않",
]

def is_refusal(response: str) -> bool:
    """Check if response is a refusal."""
    for pattern in REFUSAL_PATTERNS:
        if re.search(pattern, response):
            return True
    return False

def evaluate_refusal_accuracy(questions: list[GoldenQuestion], retriever, llm) -> dict:
    """Evaluate refusal accuracy.
    
    - out_of_scope questions: should refuse
    - Other questions: should answer
    """
    results = []
    
    for q in questions:
        response, _ = run_rag(q.question, retriever, llm)
        refused = is_refusal(response)
        should_refuse = q.q_type == 'out_of_scope'
        
        correct = (refused == should_refuse)
        
        results.append({
            'question': q.question,
            'q_type': q.q_type,
            'response': response[:100],
            'refused': refused,
            'should_refuse': should_refuse,
            'correct': correct,
        })
    
    # Calculate metrics
    df = pd.DataFrame(results)
    
    # Overall accuracy
    accuracy = df['correct'].mean()
    
    # False positive rate (오거절률): answered questions that were refused
    should_answer = df[~df['should_refuse']]
    fp_rate = should_answer['refused'].mean() if len(should_answer) > 0 else 0
    
    # True positive rate (올바른 거절률): out_of_scope that were refused
    should_refuse_df = df[df['should_refuse']]
    tp_rate = should_refuse_df['refused'].mean() if len(should_refuse_df) > 0 else 0
    
    return {
        'refusal_accuracy': accuracy,
        'false_positive_rate': fp_rate,
        'true_positive_rate': tp_rate,
        'results': df,
    }

In [ ]:
# Evaluate refusal accuracy on out_of_scope + safety + sample of others
# out_of_scope: should refuse
# safety: should answer (with safety warning)
# others: should answer

refusal_sample = [
    q for q in golden_questions 
    if q.q_type in ['out_of_scope', 'safety']
] + [q for q in golden_questions if q.q_type == 'factual'][:5]

print(f"Evaluating refusal accuracy on {len(refusal_sample)} questions...")
refusal_result = evaluate_refusal_accuracy(refusal_sample, retriever, llm)

print(f"\n=== Refusal Accuracy ===")
print(f"Overall Accuracy:     {refusal_result['refusal_accuracy']:.1%}")
print(f"True Positive Rate:   {refusal_result['true_positive_rate']:.1%} (out_of_scope correctly refused)")
print(f"False Positive Rate:  {refusal_result['false_positive_rate']:.1%} (answerable incorrectly refused)")

In [ ]:
# Show refusal results by q_type
refusal_df = refusal_result['results']
print("\n=== Refusal by q_type ===")
for q_type in refusal_df['q_type'].unique():
    subset = refusal_df[refusal_df['q_type'] == q_type]
    print(f"{q_type}: {subset['correct'].sum()}/{len(subset)} correct")

## 6. Citation Accuracy

In [ ]:
# Citation patterns: "waterpurifier_simple p.15" or "p.15" etc.
CITATION_PATTERNS = [
    r'(waterpurifier|airpurifier|vacuumcleaner)_\w+\s*p?\.?(\d+)',  # full source with page
    r'p\.?(\d+)',  # just page number
    r'페이지\s*(\d+)',
]

def extract_citations(response: str) -> list[tuple[str, str]]:
    """Extract (source, page) citations from response."""
    citations = []
    
    # Pattern 1: full source name with page
    matches = re.findall(r'((?:waterpurifier|airpurifier|vacuumcleaner)_\w+)\s*p?\.?(\d+)', response, re.IGNORECASE)
    for source, page in matches:
        citations.append((source.lower(), page))
    
    # Pattern 2: just page (assume from context)
    if not citations:
        page_matches = re.findall(r'p\.?(\d+)', response)
        for page in page_matches:
            citations.append(('unknown', page))
    
    return citations

def check_citation_accuracy(response: str, reference_context: str, docs: list[Document]) -> dict:
    """Check if citations in response match reference and retrieved docs."""
    if reference_context == 'N/A':
        # out_of_scope - should have no citation
        citations = extract_citations(response)
        return {
            'has_citation': len(citations) > 0,
            'should_cite': False,
            'correct': len(citations) == 0,
            'reason': 'out_of_scope - no citation expected',
        }
    
    # Parse reference context
    ref_match = re.search(r'(\w+)\s*p?\.?(\d+)', reference_context)
    if not ref_match:
        return {
            'has_citation': False,
            'should_cite': True,
            'correct': False,
            'reason': 'could not parse reference',
        }
    
    ref_source = ref_match.group(1).lower()
    ref_page = int(ref_match.group(2))
    
    # Extract citations from response
    citations = extract_citations(response)
    
    if not citations:
        return {
            'has_citation': False,
            'should_cite': True,
            'correct': False,
            'reason': 'no citation in response',
        }
    
    # Check if any citation matches (with ±2 page tolerance)
    for cite_source, cite_page in citations:
        cite_page = int(cite_page)
        # Source match (partial) and page within ±2
        source_match = ref_source in cite_source or cite_source in ref_source or cite_source == 'unknown'
        page_match = abs(cite_page - ref_page) <= 2
        
        if source_match and page_match:
            return {
                'has_citation': True,
                'should_cite': True,
                'correct': True,
                'reason': f'matched {cite_source} p.{cite_page} to ref {ref_source} p.{ref_page}',
            }
    
    return {
        'has_citation': True,
        'should_cite': True,
        'correct': False,
        'reason': f'citations {citations} did not match ref {ref_source} p.{ref_page}',
    }

def evaluate_citation_accuracy(questions: list[GoldenQuestion], retriever, llm) -> dict:
    """Evaluate citation accuracy."""
    results = []
    
    for q in questions:
        response, docs = run_rag(q.question, retriever, llm)
        check = check_citation_accuracy(response, q.reference_context, docs)
        
        results.append({
            'question': q.question,
            'q_type': q.q_type,
            'reference_context': q.reference_context,
            'response': response[:150],
            **check,
        })
    
    df = pd.DataFrame(results)
    
    # Only count questions that should have citations
    should_cite = df[df['should_cite']]
    accuracy = should_cite['correct'].mean() if len(should_cite) > 0 else 0
    
    return {
        'citation_accuracy': accuracy,
        'total_should_cite': len(should_cite),
        'correct_citations': should_cite['correct'].sum(),
        'results': df,
    }

In [ ]:
# Evaluate citation accuracy on sample
citation_sample = [q for q in golden_questions if q.q_type in ['factual', 'comparison', 'multi_hop', 'safety']][:10]

print(f"Evaluating citation accuracy on {len(citation_sample)} questions...")
citation_result = evaluate_citation_accuracy(citation_sample, retriever, llm)

print(f"\n=== Citation Accuracy ===")
print(f"Accuracy: {citation_result['citation_accuracy']:.1%}")
print(f"Correct: {citation_result['correct_citations']}/{citation_result['total_should_cite']}")

## 7. RAGAS 한계 사례 분석

In [ ]:
# Find cases where RAGAS score and actual quality diverge

def analyze_ragas_limits(ragas_df: pd.DataFrame) -> dict:
    """Find RAGAS limitation cases."""
    limits = {
        'high_score_bad_answer': [],  # RAGAS high but actually bad
        'low_score_good_answer': [],  # RAGAS low but actually good
    }
    
    for _, row in ragas_df.iterrows():
        faith = row['faithfulness']
        relevancy = row['response_relevancy']
        
        # High score but potentially bad answer
        # Look for cases where faithfulness is high but answer is short/incomplete
        response_len = len(row['response'])
        if faith > 0.8 and response_len < 50:
            limits['high_score_bad_answer'].append({
                'question': row['question'],
                'response': row['response'],
                'faithfulness': faith,
                'reason': 'High faithfulness but very short response - may miss details',
            })
        
        # Low score but potentially good answer
        if faith < 0.5 and response_len > 100:
            limits['low_score_good_answer'].append({
                'question': row['question'],
                'response': row['response'],
                'faithfulness': faith,
                'reason': 'Low faithfulness but detailed response - may be over-penalized',
            })
    
    return limits

In [ ]:
# Analyze RAGAS limits from the evaluation results
if 'ragas_results' in dir() and len(ragas_results) > 0:
    limits = analyze_ragas_limits(ragas_results)
    
    print("=== RAGAS 한계 사례 ===")
    print(f"\n점수 높은데 답변 나쁜 사례: {len(limits['high_score_bad_answer'])}건")
    for case in limits['high_score_bad_answer'][:2]:
        print(f"  Q: {case['question'][:50]}...")
        print(f"  A: {case['response'][:80]}...")
        print(f"  Faithfulness: {case['faithfulness']:.2f}")
        print(f"  분석: {case['reason']}")
        print()
    
    print(f"\n점수 낮은데 답변 괜찮은 사례: {len(limits['low_score_good_answer'])}건")
    for case in limits['low_score_good_answer'][:2]:
        print(f"  Q: {case['question'][:50]}...")
        print(f"  A: {case['response'][:80]}...")
        print(f"  Faithfulness: {case['faithfulness']:.2f}")
        print(f"  분석: {case['reason']}")
        print()
else:
    print("RAGAS 결과가 없습니다. 먼저 섹션 4를 실행하세요.")

## 8. Save Results

In [ ]:
# Save evaluation results
RESULTS_PATH = Path('../data/week7_evaluation_results.json')

results_summary = {
    'golden_set_size': len(golden_questions),
    'ragas': {
        'faithfulness': float(ragas_results['faithfulness'].mean()) if 'ragas_results' in dir() else None,
        'response_relevancy': float(ragas_results['response_relevancy'].mean()) if 'ragas_results' in dir() else None,
        'context_precision': float(ragas_results['context_precision'].mean()) if 'ragas_results' in dir() else None,
        'context_recall': float(ragas_results['context_recall'].mean()) if 'ragas_results' in dir() else None,
    },
    'refusal_accuracy': refusal_result['refusal_accuracy'] if 'refusal_result' in dir() else None,
    'citation_accuracy': citation_result['citation_accuracy'] if 'citation_result' in dir() else None,
}

with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
    json.dump(results_summary, f, indent=2, ensure_ascii=False)

print(f"Results saved to {RESULTS_PATH}")
print(json.dumps(results_summary, indent=2, ensure_ascii=False))

## Summary

| Metric | Score |
|--------|-------|
| Faithfulness | ? |
| Response Relevancy | ? |
| Context Precision | ? |
| Context Recall | ? |
| Refusal Accuracy | ? |
| Citation Accuracy | ? |

### RAGAS 한계 발견
1. **점수 높은데 답변 나쁜 사례**: (실행 후 채우기)
2. **점수 낮은데 답변 괜찮은 사례**: (실행 후 채우기)